In [1]:
from ax.service.ax_client import AxClient
import sys
sys.path.append('../')
import helper_functions as hf


In [3]:
iteration_to_update = 28




In [4]:
optimizer_file_path = 'iteration_' + str(iteration_to_update) + '/optimizer/optimizer_'
ax_to_update_path = optimizer_file_path + f"{iteration_to_update}_loaded.json"

ax_to_update = AxClient.load_from_json_file(ax_to_update_path)
trials_to_update = ax_to_update.get_trials_data_frame()
trials_to_update


,trial_index,arm_name,trial_status,generation_node,obj_surf_conc,Drug_MW,Drug_LogP,Drug_TPSA,surf_1,surf_1_conc,surf_2,surf_2_conc,surf_3,surf_3_conc,drug_conc
0,0,0_0,COMPLETED,None,100.0,0.2063,0.3073,0.0373,s8,12,s6,39,s7,20,25.0
1,1,1_0,COMPLETED,None,95.0,0.2063,0.3073,0.0373,s3,51,s2,4,s6,40,25.0
2,2,2_0,COMPLETED,None,100.0,0.4045,0.4196,0.0728,s5,1,s7,56,s3,0,25.0
3,3,3_0,COMPLETED,None,100.0,0.4045,0.4196,0.0728,s3,28,s3,6,s8,14,25.0
4,4,4_0,COMPLETED,None,100.0,0.2962,0.4364,0.0493,s1,27,s4,63,s5,3,25.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159,159,159_0,COMPLETED,GenerationStep_2,70.0,0.3528,0.2810,0.0711,s1,70,s3,0,s6,0,25.0
160,160,160_0,COMPLETED,GenerationStep_2,64.0,0.2063,0.3073,0.0373,s1,64,s3,0,s6,0,25.0
161,161,161_0,COMPLETED,GenerationStep_2,31.0,0.4045,0.4196,0.0728,s1,31,s3,0,s6,0,25.0
162,162,162_0,COMPLETED,GenerationStep_2,42.0,0.2962,0.4364,0.0493,s1,42,s3,0,s6,0,25.0


In [4]:
def update_data_to_optimizer(ax_client, list_of_new_failures, list_of_new_success):

    # Load the existing optimizer state
    before_update_path = optimizer_file_path + f"{iteration_to_update}_before_update.json"
    updated_path = optimizer_file_path + f"{iteration_to_update}_loaded.json"

    ax_client.save_to_json_file(before_update_path)

    # Fetch current trials
    trials_df = ax_client.get_trials_data_frame()

    if len(list_of_new_failures) != 0:
        for trial_index in list_of_new_failures:
            # Make sure we actually have this trial
            if trial_index not in trials_df["trial_index"].values:
                print(f"Trial {trial_index} not found – skipping.")
                continue

            # Build the forced-failure payload
            new_failure = {
                "obj_surf_conc": hf.surfactant_stock_conc,
            }
            # Update the trial in-place
            ax_client.update_trial_data(trial_index=trial_index, raw_data=new_failure)
            print(f"Updated trial {trial_index}: set obj_surf_conc = {hf.surfactant_stock_conc}")

    if len(list_of_new_success) != 0:
        for trial_index in list_of_new_success:
            # Make sure we actually have this trial
            if trial_index not in trials_df["trial_index"].values:
                print(f"Trial {trial_index} not found – skipping.")
                continue    

            surf_conc_success = trials_df[trials_df['trial_index'] == trial_index]['surf_1_conc'].values[0] + trials_df[trials_df['trial_index'] == trial_index]['surf_2_conc'].values[0] + trials_df[trials_df['trial_index'] == trial_index]['surf_3_conc'].values[0]
            new_success = {
                "obj_surf_conc": surf_conc_success,
            }

            # Update the trial in-place
            ax_client.update_trial_data(trial_index=trial_index, raw_data=new_success)
            print(f"Updated trial {trial_index}: set obj_surf_conc = {surf_conc_success}")

    ax_client.save_to_json_file(updated_path)
    return ax_client

In [5]:
list_of_new_failures = [127,129]


list_of_new_success = []

In [6]:
print("Please double check the results you are updating before continue...")
print("*" * 100)
print("*" * 100)
print("Changing them from SUCCESS to FAILURE")

df = trials_to_update[trials_to_update["trial_index"].isin(list_of_new_failures)]
df


Please double check the results you are updating before continue...
****************************************************************************************************
****************************************************************************************************
Changing them from SUCCESS to FAILURE


,trial_index,arm_name,trial_status,generation_node,obj_surf_conc,Drug_MW,Drug_LogP,Drug_TPSA,surf_1,surf_1_conc,surf_2,surf_2_conc,surf_3,surf_3_conc,drug_conc
127,127,127_0,COMPLETED,GenerationStep_2,100.0,0.3528,0.2810,0.0711,s1,21,s3,0,s6,7,25.0
129,129,129_0,COMPLETED,GenerationStep_2,100.0,0.4045,0.4196,0.0728,s7,46,s3,0,s1,3,25.0


In [54]:
print("Please double check the results you are updating before continue...")
print("*" * 100)
print("*" * 100)
print("Changing them from FAILURE to SUCCESS")

df = trials_to_update[trials_to_update["trial_index"].isin(list_of_new_success)]
df

Please double check the results you are updating before continue...
****************************************************************************************************
****************************************************************************************************
Changing them from FAILURE to SUCCESS


,trial_index,arm_name,trial_status,generation_node,obj_surf_conc,Drug_MW,Drug_LogP,Drug_TPSA,surf_1,surf_1_conc,surf_2,surf_2_conc,surf_3,surf_3_conc,drug_conc


In [55]:
new_ax_client = update_data_to_optimizer(ax_client = ax_to_update, list_of_new_failures = list_of_new_failures, list_of_new_success = list_of_new_success)


[INFO 07-24 15:41:20] ax.service.ax_client: Added data: {'obj_surf_conc': (100.0, None)} to trial 127.
[INFO 07-24 15:41:20] ax.service.ax_client: Added data: {'obj_surf_conc': (100.0, None)} to trial 129.


Updated trial 127: set obj_surf_conc = 100
Updated trial 129: set obj_surf_conc = 100


In [56]:
updated_trials = new_ax_client.get_trials_data_frame()
updated_trials[updated_trials["trial_index"].isin(list_of_new_failures + list_of_new_success)]

,trial_index,arm_name,trial_status,generation_node,obj_surf_conc,Drug_MW,Drug_LogP,Drug_TPSA,surf_1,surf_1_conc,surf_2,surf_2_conc,surf_3,surf_3_conc,drug_conc
127,127,127_0,COMPLETED,GenerationStep_2,100.0,0.3528,0.2810,0.0711,s1,21,s3,0,s6,7,25.0
129,129,129_0,COMPLETED,GenerationStep_2,100.0,0.4045,0.4196,0.0728,s7,46,s3,0,s1,3,25.0
